<a href="https://colab.research.google.com/github/huseyin-karaca/s2t-tr-dev/blob/main/notebooks/colab/s2t_tr_dev_ablation_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Architecture Ablation (AMI)

Reproduces the architecture-ablation table in Section 4.3 of the manuscript. Each variant changes one knob from the default hierarchical-transformer config; all variants share the same loss config (the best one from the loss ablation: `λ_wer=1.0, λ_hard=0.0, λ_soft=0.5, τ=0.1`).

The full sweep runs **8 variants** end-to-end; expect ~3–4 hours on L4 with 50 epochs each. To do a quick smoke test first, run a single variant with `--max-epochs 2` (see below).


## 1. Initial setup (clone + uv env + secrets)

GPU recommendation:
- **L4** (default): best price/performance for these notebooks. ~1.5–3× faster than T4 with no per-second penalty.
- **A100**: ~2× faster than L4, ~2× the price. Use only if you want the full 50-epoch run in under 30 minutes.
- **T4**: works but slow; halve `batch_size` if you OOM.


In [ ]:
! git clone https://github.com/huseyin-karaca/s2t-tr-dev
%cd /content/s2t-tr-dev

from google.colab import files
files.view('/content/s2t-tr-dev')

!make create_environment
!make requirements

from google.colab import userdata
import os
os.environ['GITHUB_TOKEN'] = userdata.get('GitHubPAT')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

!gh auth status
!git config --global user.name 'huseyin-karaca'
!git config --global user.email 'huseyinkaraccca@gmail.com'

from google.colab import drive
drive.mount('/content/drive')


Cloning into 's2t-tr-dev'...
remote: Enumerating objects: 368, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 368 (delta 57), reused 102 (delta 42), pack-reused 244 (from 2)
Receiving objects: 100% (368/368), 738.73 MiB | 19.32 MiB/s, done.
Resolving deltas: 100% (143/143), done.
Updating files: 100% (148/148), done.
/content/s2t-tr-dev


<IPython.core.display.Javascript object>

uv venv --python 3.10
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
>>> New uv virtual environment created. Activate with:
>>> Windows: .\.venv\Scripts\activate
>>> Unix/macOS: source ./.venv/bin/activate
uv sync
Resolved 199 packages in 11ms
Prepared 195 packages in 41.61s
Installed 195 packages in 322ms
 + absl-py==2.4.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + async-timeout==5.0.1
 + attrs==26.1.0
 + audioread==3.1.0
 + babel==2.18.0
 + beautifulsoup4==4.14.3
 + bleach==6.3.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.3.2
 + comm==0.2.3
 + contourpy==1.3.2
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.3
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + datasets==3.6.0
 + debugpy==1.8

## 2. Bring in the AMI interim features from Drive

Reuses the per-expert interim parquets you have already generated and saved to Drive (no re-extraction). Then runs the patched preprocess script that propagates the `_transcription` columns alongside the embeddings/WERs.

> **Adjust `DRIVE_INTERIM_AMI` below** to your actual Drive folder path before running.


In [ ]:
# Path on Drive that contains the three per-expert interim parquets:
#   openai_whisper-base.parquet
#   facebook_hubert-large-ls960-ft.parquet
#   facebook_wav2vec2-base-960h.parquet
DRIVE_INTERIM_AMI = '/content/drive/MyDrive/s2t-tr-dev/data/interim/edinburghcstr_ami'

!mkdir -p data/interim/edinburghcstr_ami
!cp -v "$DRIVE_INTERIM_AMI"/*.parquet data/interim/edinburghcstr_ami/
!ls -lh data/interim/edinburghcstr_ami/


In [ ]:
!uv run python -m src.data.preprocess \
    --dataset edinburghcstr/ami \
    --output_name combined_features_with_transcripts.parquet


In [ ]:
DRIVE_PROCESSED_AMI = '/content/drive/My Drive/s2t-tr-dev/data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet'

!mkdir -p data/processed/edinburghcstr_ami
!cp -v "$DRIVE_PROCESSED_AMI" /content/s2t-tr-dev/data/processed/edinburghcstr_ami/
!ls -lh data/processed/edinburghcstr_ami/


'/content/drive/My Drive/s2t-tr-dev/data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet' -> '/content/s2t-tr-dev/data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet'
total 7.5G
-rw------- 1 root root 7.5G Apr 26 19:47 combined_features-w2v2-large.parquet


## 3. Smoke test (single variant, 2 epochs)

Confirms the full pipeline before committing to the 8-variant sweep.


In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet \
    --arch hierarchical_transformer \
    --batch-size 8 --num-workers 2 \
    --max-epochs 2 --limit-batches 16 \
    --experiment-name smoke_test \
    --log-dir logs/architecture_ablation


2026-04-26 19:54:15.104 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/content/s2t-tr-dev/src/training/train.py", line 47, in <module>
    import pytorch_lightning as pl
  File "/content/s2t-tr-dev/.venv/lib/python3.10/site-packages/pytorch_lightning/__init__.py", line 25, in <module>
    from lightning_fabric.utilities.seed import seed_everything  # noqa: E402
  File "/content/s2t-tr-dev/.venv/lib/python3.10/site-packages/lightning_fabric/__init__.py", line 35, in <module>
    from lightning_fabric.fabric import Fabric  # noqa: E402
  File "/content/s2t-tr-dev/.venv/lib/python3.10/site-packages/lightning_fabric/fabric.py", line 39, in <module>
    from lightning_fabric.accelerators.accelerator import Accelera

## 4. Run the architecture ablation

8 trained variants × ~25 min each on L4 ≈ 3–4 h. The orchestrator writes one combined `ablation_results.json` after every variant, so the run is checkpoint-safe — restart and existing variants will retrain (no resume logic; the dataset and loss are identical so this is fine).

> **A100 settings**: `--max-epochs 30` (converges faster), `batch_size 16` in the JSON.
> **T4 settings**: keep epochs at 50 but set `batch_size 4` to avoid OOM.


In [ ]:
!uv run python -m src.experiments.run_ablation \
    --config configs/architecture_ablation.json \
    --output-dir reports/ablations/architecture_ami


2026-04-26 19:54:21.285 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 19:54:21,329 [INFO] __main__: ============================================================
2026-04-26 19:54:21,329 [INFO] __main__: Variant: default
2026-04-26 19:54:21,329 [INFO] __main__: ============================================================
2026-04-26 19:54:21,329 [INFO] __main__: >>> train variant=default
    /content/s2t-tr-dev/.venv/bin/python3 -m src.training.train --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet --experiment-name default --log-dir reports/ablations/architecture_ami --arch hierarchical_transformer --train-ratio 0.8 --val-ratio 0.1 --max-seq-len 2000 --batch-size 8 --num-workers 4 --max-epochs 50 --learning-rate 0.0001 --primary-weight 1.0 --aux-ce-weight 0.0 --soft-ce-weight 0.5 --soft-ce-temperature 0.1 --seed 42
2026-04-26 19:54:21.422 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-de

## 5. Render the table

Prints one row per variant (test selected_wer, oracle_wer, gap, selection_accuracy). This is the data backing the ablation table in the manuscript.


In [ ]:
import json
from pathlib import Path

results = json.loads(Path('reports/ablations/architecture_ami/ablation_results.json').read_text())
rows = []
for name, entry in results['variants'].items():
    t = entry['test']
    rows.append((name, t['selected_wer'], t['oracle_wer'], t['wer_gap'], t['selection_accuracy']))

print(f"{'variant':<22} {'sel_wer':>8} {'oracle':>8} {'gap':>8} {'sel_acc':>8}")
print('-' * 64)
for r in rows:
    print(f"{r[0]:<22} {r[1]:>8.4f} {r[2]:>8.4f} {r[3]:>8.4f} {r[4]:>8.4f}")


variant                 sel_wer   oracle      gap  sel_acc
----------------------------------------------------------------
default                  0.3158   0.2369   0.0789   0.5063
no_bridge                0.4293   0.2369   0.1924   0.4081
separate_stage1          0.3405   0.2369   0.1036   0.4921
stage1_layers_1          0.4528   0.2369   0.2158   0.4477
stage1_layers_4          0.3335   0.2369   0.0966   0.4002
stage2_layers_2          0.3923   0.2369   0.1554   0.4802
d_model_128              0.3277   0.2369   0.0908   0.4635
d_model_512              0.4221   0.2369   0.1852   0.4952


## 6. Backup logs to Drive


In [ ]:
!cp -r reports/ablations/architecture_ami /content/drive/MyDrive/s2t-tr-dev/ablations/architecture_ami_$(date +%Y%m%d_%H%M)


cp: cannot create directory '/content/drive/MyDrive/s2t-tr-dev/ablations/architecture_ami_20260427_1501': No such file or directory


In [ ]:
from google.colab import runtime
runtime.unassign()